In [1]:
import os
BASE_DIR = data_dir = "/home/salman/Documents/data/birdclef-2026/"
API_KEY = "1fe5aa16c7200787a6cd575a9a52abf0e3d5085d"

In [2]:
import pandas as pd
tax = pd.read_csv(os.path.join(BASE_DIR, "taxonomy.csv"))
tax.head()

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia


In [3]:
import xenocanto
import asyncio

In [4]:
dir = "outputs"
os.makedirs(dir, exist_ok=True)



In [5]:
from xcapi.query import QueryBuilder
from xcapi.client import XenoCantoClient
from xcapi.downloader import Downloader
from tqdm import tqdm
import requests
from concurrent.futures import ThreadPoolExecutor
import os
import requests
from tqdm import tqdm
import os
import requests
from urllib.parse import urlparse, unquote


# def download_one(_rec):
#     new_dir, rec = _rec
#     save_name = os.path.join(new_dir, rec['file-name'])
#     if os.path.exists(save_name):
#         return True  # already downloaded
#     try:
#         response = requests.get(rec['file'], timeout=300)
#         if response.status_code == 200:
#             with open(save_name, 'wb') as f:
#                 f.write(response.content)
#             return True
#         return False
#     except Exception:
#         return False


def download_one(_rec):
    new_dir, rec = _rec

    try:
        response = requests.get(rec['file'], timeout=300, stream=True)

        if response.status_code != 200:
            return False

        # 1. Try Content-Disposition header
        filename = None
        cd = response.headers.get("Content-Disposition")

        if cd and "filename=" in cd:
            filename = cd.split("filename=")[-1].strip('"')

        # 2. Fallback to filename from URL
        if not filename:
            path = urlparse(response.url).path
            filename = os.path.basename(path)
            filename = unquote(filename)

        # 3. Final fallback
        if not filename:
            filename = rec.get('file-name', 'downloaded_file')

        save_name = os.path.join(new_dir, filename)

        if os.path.exists(save_name):
            return True

        with open(save_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        return True

    except Exception:
        return False

In [6]:
# import os
# import requests

# def download_one(_rec, retries=3):
#     new_dir, rec = _rec
#     save_name = os.path.join(new_dir, rec["file-name"])
#     if os.path.exists(save_name):
#         return True
#     tmp = save_name + ".part"

#     for attempt in range(retries):
#         try:
#             with requests.get(rec["file"], timeout=300, stream=True) as r:
#                 if r.status_code != 200:
#                     continue
#                 ct = r.headers.get("Content-Type", "").lower()

#                 expected = r.headers.get("Content-Length")
#                 expected = int(expected) if expected and expected.isdigit() else None

#                 written = 0
#                 with open(tmp, "wb") as f:
#                     for chunk in r.iter_content(chunk_size=64 * 1024):
#                         if chunk:
#                             f.write(chunk)
#                             written += len(chunk)

#                 if expected is not None and written != expected:
#                     os.remove(tmp)
#                     continue

#                 os.replace(tmp, save_name)
#                 return True
#         except requests.RequestException as e:
#             if os.path.exists(tmp):
#                 os.remove(tmp)
            
#             if attempt == retries - 1:
#                 return (False, str(e))

#     return False

In [7]:
client = XenoCantoClient(api_key=API_KEY)

In [8]:
query = QueryBuilder().latitude("<-16").latitude(">-22.1").longitude("<-55.4").longitude(">-58.1").build()
site_recordings = client.search(query)
print (len(site_recordings))

2015


In [9]:
species_recordings = []
for i, row in tqdm(tax.iterrows(), total=len(tax)):
    query = QueryBuilder().species(row['scientific_name'])
    query = query.build()
    recordings = client.search(query)
    if len(recordings) > 1:
        species_recordings.extend(recordings)
print (len(species_recordings))

100%|██████████| 234/234 [05:19<00:00,  1.37s/it]

39172


In [10]:
total_recordings = []
total_recordings.extend(site_recordings)
total_recordings.extend(species_recordings)
print (len(total_recordings))

41187


In [11]:
import json

seen = set()
unique = []

for d in total_recordings:
    key = json.dumps(d, sort_keys=True)
    if key not in seen:
        seen.add(key)
        unique.append(d)

In [12]:
unique_recordings = unique

In [13]:
len(unique_recordings)

39924

In [14]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

results = []

with ThreadPoolExecutor(max_workers=24) as executor:
    futures = [
        executor.submit(download_one, (dir, rec))
        for rec in unique_recordings
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        results.append(future.result())

100%|██████████| 39924/39924 [50:52<00:00, 13.08it/s]  


In [17]:
len(os.listdir("outputs"))

23459

In [18]:
import numpy as np
exts = []
for audio in os.listdir("outputs"):
    exts.append(audio.split(".")[-1])
np.unique(exts)

array(['', ' freile', 'mp3', 'wav'], dtype='<U7')

In [1]:
# NEW_DIR = "outputs_ogg"
# os.makedirs(NEW_DIR, exist_ok=True)
# from pydub import AudioSegment
# for audio in tqdm(os.listdir("outputs"), total=len(os.listdir("outputs"))):
    

#     xaudio = ".".join(audio.split(".")[:-1])+".ogg"
#     if os.path.exists(os.path.join(NEW_DIR, xaudio)):
#         continue

    
#     other_ext_fp = os.path.join("outputs", audio)

#     try:

#         if audio.split(".")[-1].lower() == "wav":
#             song = AudioSegment.from_file(other_ext_fp)
#         elif audio.split(".")[-1].lower() == "mp3":
#             song = AudioSegment.from_file(other_ext_fp)
#         elif audio.split(".")[-1].lower() == "freile":
#             song = AudioSegment.from_file(other_ext_fp)
#         else:
#             print ("Invalid")
#             continue

#         song = song.set_frame_rate(32000)
#         song.export(os.path.join(NEW_DIR, xaudio), format="ogg")
#     except:
#         continue

In [9]:
from concurrent.futures import ProcessPoolExecutor
from pydub import AudioSegment
from functools import partial
INPUT_DIR = "outputs"
NEW_DIR = "outputs_ogg"

os.makedirs(NEW_DIR, exist_ok=True)


def convert_audio(audio_file, input_dir, output_dir):
    try:
        ext = audio_file.split(".")[-1].lower()

        if ext not in {"wav", "mp3", "freile"}:
            return f"Skipped: {audio_file}"

        xaudio = ".".join(audio_file.split(".")[:-1]) + ".ogg"
        output_fp = os.path.join(output_dir, xaudio)

        if os.path.exists(output_fp):
            return f"Exists: {xaudio}"

        input_fp = os.path.join(input_dir, audio_file)

        song = AudioSegment.from_file(input_fp)
        song = song.set_frame_rate(32000)

        song.export(output_fp, format="ogg")

        return f"Done: {xaudio}"

    except Exception as e:
        return f"Error ({audio_file}): {e}"

In [10]:
audio_files = os.listdir(INPUT_DIR)

worker = partial(
    convert_audio,
    input_dir=INPUT_DIR,
    output_dir=NEW_DIR,
)

with ProcessPoolExecutor() as executor:
    results = list(
        tqdm(
            executor.map(worker, audio_files),
            total=len(audio_files),
        )
    )

100%|██████████| 23459/23459 [03:37<00:00, 107.94it/s]  


In [11]:
len(os.listdir(NEW_DIR))

23454

In [ ]:
len(unique_recordings)

39922

In [12]:
!pwd

/home/salman/Documents/codebases/ADD_DATA_XC_SITE
